In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/processed/cleaned_clients.csv")

df.head

In [2]:
df.shape

In [3]:
df.dtypes

In [4]:
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"]

In [5]:
y = y.map({
    "No": 0,
    "Yes": 1
})

In [6]:
y.value_counts()

In [7]:
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 6, 12, 24, 48, 72],
    labels=[
        "0-6 months",
        "7-12 months",
        "13-24 months",
        "25-48 months",
        "49-72 months"
    ]
)

In [8]:
df["AvgMonthlySpend"] = np.where(
    df["tenure"] > 0,
    df["TotalCharges"] / df["tenure"],
    df["MonthlyCharges"]
)

In [9]:
service_columns = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

In [10]:
df["ServiceCount"] = (
    df[service_columns]
    .replace({
        "Yes": 1,
        "No": 0,
        "No internet service": 0,
        "No phone service": 0
    })
    .sum(axis=1)
)

In [11]:
df["ServiceCount"] = (
    df[service_columns]
    .replace({
        "Yes": 1,
        "No": 0,
        "No internet service": 0,
        "No phone service": 0
    })
    .sum(axis=1)
)

In [12]:
df[["tenure", "TotalCharges", "MonthlyCharges",
    "AvgMonthlySpend", "ServiceCount"]].head()

In [13]:
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [14]:
print("X shape:", X.shape)
print("y shape:", y.shape)

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [16]:
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining churn distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting churn distribution:")
print(y_test.value_counts(normalize=True))

In [17]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

In [18]:
print("Missing values in training features:")
print(X_train.isnull().sum().sort_values(ascending=False).head(10))

print("\nMissing values in testing features:")
print(X_test.isnull().sum().sort_values(ascending=False).head(10))

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Numeric features
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical features
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both transformers
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print("Preprocessing pipeline created successfully!")

In [20]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

In [21]:
%pip install xgboost

In [22]:
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

In [23]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=5174 / 1869,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

In [24]:
results = []
trained_models = {}

for name, model in models.items():

    print(f"Training {name}...")

    pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
        "PR-AUC (AP)": average_precision_score(y_test, y_proba)
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values(
    by="PR-AUC (AP)",
    ascending=False
)

results_df.round(4)

In [25]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

best_model = trained_models["XGBoost"]

y_pred_xgb = best_model.predict(X_test)

print("Classification Report:")
print(classification_report(
    y_test,
    y_pred_xgb,
    target_names=["No Churn", "Churn"],
    digits=4
))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_xgb,
    display_labels=["No Churn", "Churn"],
    cmap="Blues"
)

plt.title("XGBoost - Confusion Matrix")
plt.tight_layout()
plt.show()

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=2cb50308-81e6-4757-a2d6-155b4c4296dc' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>